In [1]:
!pip install ultralytics opencv-python pymongo pyserial



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pymongo import MongoClient
from datetime import datetime

client = MongoClient("mongodb://localhost:27017/")
db = client["object_detection_db"]
collection = db["detections"]

collection.insert_one({
    "object": "test_object",
    "confidence": 1.0,
    "time": datetime.now()
})

print(db.list_collection_names())



['detections']


In [3]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # smallest model


In [5]:
import cv2

cap = cv2.VideoCapture(2)  # external webcam (change to 0 if needed)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLOv8 object detection happens HERE
    results = model(frame, verbose=False)

    # Draw bounding boxes (optional but good for demo)
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[cls_id]

            if conf > 0.6:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    frame,
                    f"{label} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

    cv2.imshow("YOLOv8 Object Detection", frame)

    # PRESS 'q' TO STOP
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [4]:
import serial
import time

ser = serial.Serial("COM9", 9600)
time.sleep(2)


In [ ]:
import cv2

cap = cv2.VideoCapture(2)  # external webcam index

last_sent = ""

allowed_objects = ["person", "bottle", "book"]  # move outside loop for efficiency

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            label = model.names[cls_id]

            if conf > 0.6 and label in allowed_objects:
                # Avoid spamming same result
                if label != last_sent:
                    last_sent = label

                    # Save to MongoDB
                    collection.insert_one({
                        "object": label,
                        "confidence": conf,
                        "time": datetime.now()
                    })

                    # Send to ESP32
                    ser.write((label + "\n").encode())

                # Draw bounding box + label on the frame
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    frame,
                    f"{label} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 0),
                    2
                )

    cv2.imshow("YOLOv8 Detection", frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break



: 